# 5 Your own search, and a negative result you produced yourself

Notebook 04 ran one search and read it for you. Here you drive.

**The deliverable is a short report** — half a page is plenty — containing at least one
**negative result that you produced and are willing to stand behind**. Not a failure you
apologise for: a hypothesis you stated, tested cheaply, and found did not pay. That habit
is the entire reason this lesson exists, and it is the thing being graded.

A report that says "I tried six things and here is the best one" is worth less than one
that says "I expected X, measured it, got Y, and here is the seed spread that makes me
believe it".

Everything runs on **DTD** — 47 texture classes, roughly twenty points of headroom, and an
encoder axis that actually moves.

In [ ]:
import os
import warnings

os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")     # see notebook 04
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

import sys
import time
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.components import GaussianNoise, NeuralNet, Serial
from vectormesh.components.metrics import Accuracy
from vectormesh.data import Collate, OneHot
from vectormesh.training import Trainer, TrainerSettings

from notebooktester import param
from runlog import log_run, use_experiment

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

EPOCHS = param(15, test=1)
N_TRIALS = param(16, test=2)
SEEDS = param(3, test=2)

ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

# Every run below is logged through runlog.log_run, in the shape the class run protocol
# expects: you submit these runs at the end of the notebook, and the next lesson opens
# with the whole class's runs side by side.
use_experiment("lesson1-exercises")

In [ ]:
def load(encoder: str, split: str) -> VectorCache:
    return VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split=split)


def loaders(encoder: str, batch_size: int = 64, n_train: int | None = None):
    # n_train shrinks the training set for the data-size questions below.
    # The shuffle is NOT optional -- see the cell after this one.
    out = []
    for split in ("train", "test"):
        cache = load(encoder, split)
        ds = cache.dataset
        if n_train is not None and split == "train":
            ds = ds.shuffle(seed=0).select(range(n_train))
        oh = OneHot(num_classes=cache.num_classes, label_col="label", target_col="onehot")
        col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
        out.append(DataLoader(ds.map(oh), batch_size=batch_size,
                              shuffle=(split == "train"), collate_fn=col))
    return out


def run(model, train_dl, test_dl, lr: float = 1e-3, epochs: int = EPOCHS) -> float:
    # vectormesh.training.Trainer, same as notebook 04: log_dir is computed but never
    # created on disk unless something writes to it, and progress=False drops the tqdm
    # bars, which is what makes calling this once per seed -- several times per exercise
    # below -- cheap. `.eval()`/`.train()`/`no_grad()` are handled internally too.
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(train_dl),
        valid_steps=len(test_dl),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=train_dl,
        validdataloader=test_dl,
        progress=False,
    )
    return trainer.loop().metric_dict["Accuracy"]


def make_head(config: dict, dim: int, n_classes: int) -> Serial:
    # the same builder as notebook 04 §4.1: 0 hidden layers is the baseline's single Linear,
    # more layers go through one NeuralNet call, which puts a GELU between every Linear.
    # Add an axis to param_space? Use it here.
    n_hidden = config["n_hidden_layers"]
    if n_hidden == 0:
        head = nn.Linear(dim, n_classes)
    else:
        head = NeuralNet([dim, *[config["hidden_dim"]] * n_hidden], n_classes)
    return Serial([GaussianNoise(sigma=config["sigma"]), head])


def dim_of(encoder: str) -> int:
    return DIM[encoder] if encoder in DIM else load(encoder, "train").dim


def seed_spread(make_model, train_dl, test_dl, seeds: int = SEEDS, **kwargs) -> np.ndarray:
    out = []
    for seed in range(seeds):
        torch.manual_seed(seed)
        out.append(run(make_model(), train_dl, test_dl, **kwargs))
    return np.array(out)


def report(name: str, scores: np.ndarray, versus: np.ndarray | None = None) -> None:
    line = f"{name:<32} {scores.mean():.4f} +/- {scores.std():.4f}   (n={len(scores)})"
    if versus is not None:
        diff = scores.mean() - versus.mean()
        pooled = np.sqrt(scores.var(ddof=1) / len(scores) + versus.var(ddof=1) / len(versus))
        line += f"   {100 * diff:+.2f} pt, {diff / pooled:+.1f} sigma"
    print(line)


DIM = {enc: load(enc, "train").dim for enc in ENCODERS}
N_CLASSES = load("dinov2-small", "train").num_classes
print("widths:", DIM, "| classes:", N_CLASSES)

### First, look at the data you are about to slice

That `.shuffle(seed=0)` above is not defensive habit, it is load-bearing, and the reason is
worth two minutes because it is the kind of bug that produces a *plausible* wrong answer
rather than a crash.

In [ ]:
dtd = load("dinov2-small", "train")
labels = dtd["label"][:]
shuffled_labels = dtd.dataset.shuffle(seed=0)["label"][:]

print("first 20 labels in the DTD cache:", labels[:20].tolist())
print()
print(f"{'rows taken':>12} {'classes, as stored':>20} {'classes, shuffled':>19}")
for n in (470, 940, 1880):
    print(f"{n:>12} {len(set(labels[:n].tolist())):>14}/{N_CLASSES:<5} "
          f"{len(set(shuffled_labels[:n].tolist())):>13}/{N_CLASSES}")

**The DTD cache is stored grouped by label.** Take the first 940 rows and you do not get a
smaller sample of the task — you get a *different, easier* task with a quarter of the
classes, and a test set full of categories your model has never once seen.

The reason this is dangerous rather than merely wrong is that it fails *quietly and
plausibly*. Subsampling this way at 470 / 940 / 1880 rows produces test accuracies of
0.12 / 0.24 / 0.44, which looks exactly like a healthy learning curve — right up until you
notice those numbers are just `n / 3760`. The model was scoring near-perfectly on the
classes it had and zero on the rest. Shuffled, the same sizes give about 0.69 / 0.75 / 0.78
(measured with a one-hidden-layer head; `planning/evidence.md` §3b), which
is the real curve and a far less exciting one.

Two general lessons, both cheap:

- **Not every cache is shuffled.** The EuroSAT caches are (they were subsampled at build
  time, which shuffles); the DTD ones were built `--full`, which does not. Do not assume —
  it is one line to check, as above.
- **A number that looks like a textbook result deserves more suspicion, not less.** The
  broken version of this exercise produced a *nicer* curve than the correct one.

## E1 — The baseline, first

**Before you search anything**, establish the two-line baseline on DTD and its seed
spread. A search without a baseline is unreadable; a baseline without a spread cannot tell
you whether a later "win" is real.

This one is done for you, because it is the habit the rest of the notebook depends on.
Every number you produce below gets compared against these two.

In [ ]:
train_dl, test_dl = loaders("dinov2-small")

baseline = seed_spread(lambda: nn.Linear(DIM["dinov2-small"], N_CLASSES), train_dl, test_dl)
report("baseline: Linear(384, 47)", baseline)
print(f"\n=> a later result must clear {baseline.mean() + 2 * baseline.std():.4f} "
      f"before you should believe it beat this.")

example = next(iter(train_dl))[0][:2]   # two rows are enough to trace every layer's shape
log_run("baseline", "baseline", scores={"accuracy": baseline},
        params={"encoder": "dinov2-small", "epochs": EPOCHS},
        model=nn.Linear(DIM["dinov2-small"], N_CLASSES), example=example)

**Write down both numbers now**, in your report, before you have seen anything else. This
is not ceremony — it is the only way to stop yourself from quietly redefining what counted
as success once you have seen the results.

## E2 — Argue from the metadata, then check

Notebook 01 said the metadata is the public API of a cache. This is not a cold guess —
you already have evidence, so use it:

**Before running the next cell**, answer in your report:

1. Look back at notebook 01 §1.5's table: DINOv2 beat resnet-18 and mobilenet by 9-15
   points on *flowers*. Textures are a different domain, but the *ranking* of general-purpose
   image encoders tends to be fairly stable across natural-image tasks. Would you expect the
   same ordering here, or is there a reason textures specifically might reshuffle it? Either
   answer is defensible — the point is to have reasoned about it, not to have guessed right.
2. `mobilenet_v2` produces 1280 numbers per image against DINOv2's 384. Notebook 01 §1.5
   already told you width is not quality on flowers (`mobilenet_v2` lost by eleven points
   there despite more dimensions). Do you expect that to hold here too?
3. On flowers the encoder spread was about 15 points and on EuroSAT about 3. DTD's spread is
   somewhere in that range — pick a number and say why textures should be closer to one end
   than the other.

Then run it.

In [ ]:
results = {}
for enc in ENCODERS:
    tr, te = loaders(enc)
    torch.manual_seed(0)
    results[enc] = run(nn.Linear(DIM[enc], N_CLASSES), tr, te)

print(f"{'encoder':<24} {'width':>6} {'accuracy':>9}")
print("-" * 42)
for enc, acc in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"{enc:<24} {DIM[enc]:>6} {acc:>9.4f}")

spread = max(results.values()) - min(results.values())
print(f"\nspread across encoders: {100 * spread:.1f} pt")
print(f"for comparison, the baseline's own seed spread: {100 * baseline.std():.2f} pt")

Compare that last pair of numbers. **How many seed-spreads wide is the encoder axis?** That
ratio is the honest way to say "this axis matters and that one does not", and it is the
sentence your report should contain.

## E3 — Design and run your own search

Now your search space. Notebook 04's was handed to you; this one is yours to defend.

Edit `param_space` below. Some things worth deciding, and *writing down the reasoning for*:

- **Should `encoder` be in the space at all?** You have just measured that axis directly.
  Leaving it in means most of your budget gets spent re-confirming what you already know.
  Taking it out means your search can only explore the flat part. Which is the better use
  of 16 trials? There is a defensible answer either way — pick one and say why.
- **What is the widest hidden layer worth trying?** Notebook 04's evidence says added depth
  loses on a frozen encoder. Is your space designed to *find* a good architecture, or to
  *test* that claim? Those want different ranges.
- **How wide a learning-rate range?** Too narrow and you have assumed the answer; too wide
  and you burn trials on configs that cannot train.

That is the whole knob-set this notebook gave you, and it is worth being explicit that it
is not the only one. A few ways to widen the design space, none required, all cheap to try:

- **A `Parallel` fusion branch**, notebook 04 §4.3's construction, as one more entry in the
  space instead of a separate experiment — does giving the search the *option* of fusion
  change what it picks, compared to deciding by hand?
- **More encoders than the three published here.** Every one you have used so far was
  chosen for you. If you want to try a fourth — a CLIP variant, a newer DINOv3, a
  domain-specific encoder — an LLM with web or an MCP tool for browsing the HuggingFace Hub
  can shortlist candidates faster than reading model cards one by one; a defensible
  rationale still has to be *yours* (what was it pretrained on, and does that resemble
  textures?). Once you have picked one, `01c_build_your_own_cache.ipynb` (optional, in this
  same folder) builds a small cache for it on any hardware.
- **A different kind of head knob entirely** — dropout, a residual connection, weight decay.
  None of these were in the space you were handed, which was itself a choice, not a law.

You are not required to use any of these. The point is that the search space you were given
was a design decision someone made, not the only one available — naming what you would add
or remove, and why, is worth more in the report than running a bigger search.

In [ ]:
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch


def trainable(config):
    encoder = config["encoder"]
    train = VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split="train")
    test = VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split="test")
    n_classes = train.num_classes
    dim = train.dim

    oh = OneHot(num_classes=n_classes, label_col="label", target_col="onehot")
    col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
    tr = DataLoader(train.dataset.map(oh), batch_size=64, shuffle=True, collate_fn=col)
    te = DataLoader(test.dataset.map(oh), batch_size=64, shuffle=False, collate_fn=col)

    model = make_head(config, dim, n_classes)

    # report every epoch, not only the last, so each trial's learning curve reaches mlflow
    def to_ray(epoch, train_loss, test_loss, metric_dict):
        tune.report({"Accuracy": metric_dict["Accuracy"], "train_loss": train_loss, "test_loss": test_loss})

    torch.manual_seed(0)
    settings = TrainerSettings(
        epochs=config["epochs"],
        metrics=[Accuracy()],
        logdir=Path("ray_results/trials").resolve(),  # never created unless earlystop saves
        train_steps=len(tr),
        valid_steps=len(te),
        optimizer_kwargs={"lr": config["lr"]},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=tr,
        validdataloader=te,
        progress=False,  # ray's own status table already shows progress across trials
        reporters=[to_ray],
    )
    trainer.loop()


# ---- YOUR SEARCH SPACE. Edit this, and justify every line of it in your report. ----
param_space = {
    "encoder": tune.choice(["dinov2-small"]),
    "n_hidden_layers": tune.choice([0, 1]),
    "hidden_dim": tune.choice([64, 256]),   # ignored by trainable when n_hidden_layers=0
    "sigma": tune.uniform(0.0, 0.5),
    "lr": tune.loguniform(3e-4, 3e-3),
    "epochs": EPOCHS,
}

In [ ]:
tuner = tune.Tuner(
    trainable,
    param_space=param_space,
    tune_config=tune.TuneConfig(metric="Accuracy", mode="max",
                                search_alg=HyperOptSearch(), num_samples=N_TRIALS),
    run_config=tune.RunConfig(
        storage_path=Path("ray_results").resolve(),
        name="my_search",
        # verbose=1, no explicit reporter: ray auto-detects a running Jupyter kernel and
        # renders the same live status table 4_tuning_networks/01_hypertuner.ipynb gets.
        verbose=1,
    ),
)
start = time.time()
search = tuner.fit()
df = search.get_dataframe()
df = df[["Accuracy"] + [c for c in df.columns if c.startswith("config/") and "epochs" not in c]]
df.columns = [c.replace("config/", "") for c in df.columns]

print(f"{N_TRIALS} trials in {time.time() - start:.0f}s\n")
print(df.sort_values("Accuracy", ascending=False).head(6).to_string(index=False))
print(f"\nbest trial : {df['Accuracy'].max():.4f}")
print(f"baseline   : {baseline.mean():.4f} +/- {baseline.std():.4f}")

In [ ]:
# Every trial becomes its own mlflow run, with its learning curve and its architecture.
# They are logged here, after the search, and not from inside the trials: ray runs each
# trial in its own process, and the sqlite store wants one writer at a time.
logged = 0
for i, result in enumerate(search):
    if result.error or result.metrics_dataframe is None:
        continue
    config = dict(result.config)
    curves = result.metrics_dataframe[["Accuracy", "train_loss", "test_loss"]]
    dim = dim_of(config["encoder"])
    log_run(f"trial-{i:02d}", "trial", params=config,
            curves={"accuracy": curves["Accuracy"].tolist(),
                    "train_loss": curves["train_loss"].tolist(),
                    "test_loss": curves["test_loss"].tolist()},
            model=make_head(config, dim, N_CLASSES), example=torch.zeros(2, dim))
    logged += 1
print(f"logged {logged} trials to mlflow, experiment lesson1-exercises")

In [ ]:
from plotting import search_parallel_plot

# Same plot as notebook 04 §4.2, generalised to whatever columns YOUR param_space
# produced -- one line per trial, every axis at once, so an interaction between two axes
# is still visible even though no single-axis panel would show it.
fig = search_parallel_plot(df, columns=list(df.columns), color="Accuracy")
fig.update_layout(title="Your search, every trial and every axis at once")
fig.show()

**Did anything beat the baseline? By how much?**

Careful with the comparison you just printed. The best trial is a **single seed**, selected
as the maximum of 16 draws. The baseline is a **mean over several seeds**. Taking the max
of many noisy numbers and comparing it to an average is one of the most common ways to
manufacture a win that is not there — the max of 16 draws from the baseline's own
distribution would already sit above the baseline's mean.

Which is exactly what E4 is for.

## E4 — Rerun the winner honestly

Take the winning configuration and rerun it across seeds. Then compare like with like.

In [ ]:
best = search.get_best_result().config
print("winning config:", {k: v for k, v in best.items() if k != "epochs"})


def make_winner():
    return make_head(best, dim_of(best["encoder"]), N_CLASSES)


win_tr, win_te = loaders(best["encoder"])
winner = seed_spread(make_winner, win_tr, win_te, lr=best["lr"])

print()
report("baseline", baseline)
report("search winner", winner, versus=baseline)
print(f"\nsingle-seed best from the search : {df['Accuracy'].max():.4f}")
print(f"the same config over {SEEDS} seeds     : {winner.mean():.4f}")

In [ ]:
# The number this notebook is graded on: the winner rerun over seeds, not its single-seed
# leaderboard score. In the class's mlflow it sits next to everyone else's.
log_run("winner", "final", params=best, scores={"accuracy": winner},
        model=make_winner(), example=torch.zeros(2, dim_of(best["encoder"])))

**This is the number that goes in your report**, not the leaderboard number.

Answer these in writing:

1. How much of the apparent win survived being rerun? If it shrank, say by how much.
2. Is the difference larger than the pooled sigma printed above? If not, what is the honest
   one-sentence conclusion?
3. Most "wins" at this scale are inside seed noise. Is yours? Commit to an answer.

## E5 — An ablation you can actually settle

`GaussianNoise` is the one head-side knob with any prior evidence behind it: earlier
measurements had it worth about +0.4 points on DTD. That is small enough that it is a
genuinely open question rather than a known result.

**Predict first:** does input noise still help once the training set is full size? Argue
from what noise is *for* — then test it at two training sizes.

In [ ]:
FULL = 3760


def with_noise(sigma):
    return lambda: Serial([GaussianNoise(sigma=sigma),
                           nn.Linear(DIM["dinov2-small"], N_CLASSES)])


def train_accuracy(model, train_dl) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in train_dl:
            correct += int((model(X).argmax(1) == y.argmax(1)).sum())
            total += len(y)
    return correct / total


for n_train in (param(940, test=200), None):
    tr, te = loaders("dinov2-small", n_train=n_train)
    rows = n_train or FULL
    # scale epochs so every training size gets a comparable number of gradient steps,
    # otherwise you are measuring "less data" and "less training" at the same time
    epochs = max(EPOCHS, round(EPOCHS * FULL / rows))

    print(f"\n--- {rows} rows, {epochs} epochs ---")
    clean = seed_spread(with_noise(0.0), tr, te, epochs=epochs)
    noisy = seed_spread(with_noise(0.3), tr, te, epochs=epochs)
    report("sigma = 0.0", clean)
    report("sigma = 0.3", noisy, versus=clean)
    for sigma, scores in ((0.0, clean), (0.3, noisy)):
        log_run(f"noise-{sigma}-{rows}rows", "ablation", scores={"accuracy": scores},
                params={"encoder": "dinov2-small", "sigma": sigma, "n_train": rows, "epochs": epochs},
                model=with_noise(sigma)(), example=torch.zeros(2, DIM["dinov2-small"]))

    torch.manual_seed(0)
    fitted = with_noise(0.0)()
    run(fitted, tr, te, epochs=epochs)
    print(f"{'training-set accuracy':<32} {train_accuracy(fitted, tr):.4f}")

Note the epoch scaling in that cell. Without it you would be changing two things at once —
less data *and* fewer gradient steps — and could not attribute the difference to either.
Say in your report whether you think that was the right call.

Three things to take from the output, and the third is the one worth keeping.

**Check whether either difference cleared its sigma.** If it did not, the honest sentence is
"no measurable effect at this training size", not "noise helped slightly". Resist the pull
of the sign: a +0.1 pt difference at 0.4 sigma is not a small benefit, it is no evidence.

**"Does GaussianNoise help?" is not a question with an answer.** "Does it help at *this*
training size, on *this* representation?" is. Regularisation is supposed to buy something
exactly when a model can memorise its training set — so look at the last line.

**The head reaches ~100% training accuracy and the test number does not move.** That is the
informative result. The model is memorising completely, and regularising it harder changes
nothing downstream, which means the ceiling you are hitting is not head capacity — it is
what the frozen representation makes linearly separable. You cannot regularise your way to
a better representation. Notebook 04 reached the same conclusion from the other direction:
the encoder moves accuracy by more than ten points, the head knobs by about two. It is worth
writing in your report in those terms.

## E6 — Your own hypothesis

One more, of your own choosing. Something you can state as a claim that could fail, build
in a few lines, and measure against the baseline you established in E1. Some that are
cheap and genuinely open:

- Does the encoder ranking from E2 hold on a *different* dataset (`pttrn-io/eurosat-*`), or
  is it a property of textures specifically?
- Does fusion (notebook 04 §4.3) behave differently when the two encoders are close in
  strength — say `resnet-18` + `mobilenet_v2` — than when one clearly dominates?
- Does the gap between the encoders widen or narrow as you shrink the training set? Which
  direction did you expect, and why?
- One hidden layer beats the linear baseline by about two points on DTD. Does that gain hold
  on EuroSAT, where the linear head already scores around 0.94?

Pick one. State it so it could fail. Measure it over seeds. Log each arm with the same
helper the cells above use, so it reaches the class with everything else:

```python
log_run("my-hypothesis-control", "hypothesis", scores={"accuracy": control},
        params={"claim": "fusion helps when the encoders are close"},
        model=make_control(), example=example)
```

---

## The deliverable

Two parts: your runs, and a short presentation.

### 1. Submit your runs before the next lesson

Every cell above logged its runs to mlflow, in the experiment `lesson1-exercises`: the
baseline, each trial of your search with its learning curve, the winner rerun over seeds,
the noise ablation, and the arms of your own hypothesis. Each run carries its
architecture, with the tensor shape at every layer, in its `architecture.json`.

Send them to the class runs repository. From the root of the course repository:

```bash
uv run scripts/submit_runs.py --lesson 1 --list   # what would be submitted
uv run scripts/submit_runs.py --lesson 1          # submit it
```

Or tell your coding agent: *"follow docs/submit-runs.md for lesson 1"*. That file walks it
through the one-time setup (cloning the runs repository next to this one) and the
submission. Submitting again later is safe; only new runs are added.

The next lesson opens with the whole class's runs in one mlflow: which architectures were
tried, what they scored against their parameters, and how much of each search winner
survived being rerun over seeds. Your runs are part of that picture, including the ones
that lost.

### 2. A short presentation

Build it from what is in mlflow rather than retyping numbers by hand. Open your own runs
with `uv run mlflow ui --backend-store-uri sqlite:///artefacts/mlflow/mlruns.db` (from the
root of the course repository) and select the `lesson1-exercises` experiment.

1. **The baseline**, with its seed spread, from E1 — established *before* anything else.
2. **Your search space, and why.** A screenshot or sketch of `param_space` is fine; the
   reasoning behind each axis (kept in / left out / added) is the part that matters, not the
   dict itself.
3. **What the search found**, compared honestly: the winner rerun over seeds (the `winner`
   run), not the single-seed leaderboard maximum.
4. **At least one negative or null result you produced**, with its numbers. "I expected X,
   measured Y, and here is the spread that convinces me the difference is not real" is a
   complete and respectable finding, and it presents just as well as a win does.
5. **A picture of what you built.** A block diagram of your winning `Serial`/`Parallel`
   composition. The `architecture.json` of the `winner` run lists every layer with its
   input and output shape; hand-draw it, or give that file to an LLM and ask for a simple
   boxes-and-arrows diagram. A diagram communicates a design choice faster than a config
   dict does.
6. **One sentence on what you would measure next**, and why that is the question with the
   most leverage on it.

What is being graded is point 4 and the honesty of point 3. A presentation where everything
worked is one where nothing was risked — and, more often than not, one where the seed spread
was never checked.